# Đối chứng Graph trên 120 câu lao động

**Phạm vi:** chỉ `graph.enabled` khác nhau. Giữ BM25, Dense, reranker, Researcher, Auditor, Adjudicator, hậu kiểm và các kênh truy xuất chung. Giữ các đường tắt tra cứu chính xác của cấu hình gốc; không ép Graph chạy ở câu đã đủ căn cứ.

- Một lần Run All = một nhánh. Lần 1 `ARM='graph_on'`; lần 2 `ARM='graph_off'`. Giữ nguyên code ZIP/commit, artifact, model, Gold, phần cứng và biến khác.
- Mặc định `SOURCE='attached_zip'`: thêm **Graph_Study_Code.zip** (hoặc Dataset Kaggle đã giải nén gói này) và **một artifact vận hành** vào Input; không thêm đồng thời artifact cũ và mới. Nếu dùng GitHub, cần push folder mới trước rồi ghim cùng CODE_REF SHA cho cả hai lần.
- GPU T4 x2, Internet ON. Secrets: `GEMINI_API_KEY`, tùy chọn `HF_TOKEN`. Backend Gemini 3.1, judge Gemini 3.5 là hai vai trò khác nhau, giống nhau giữa hai nhánh.
- Chạy smoke để kiểm tra quy trình rồi standard. Smoke chỉ 5 câu cố định, không dùng kết luận về Graph.
- Tải ZIP ngay sau mỗi lần. Nếu quota/lỗi: giữ checkpoint, dùng cùng code/model/dữ liệu để resume; không xoay key tự động.
- `GRAPH_SCOPE=focused` chạy 47 câu (7 quan hệ + 16 điều kiện + 12 đối chứng một điều + 12 xử lý hành vi). `all120` chạy đủ 120 câu.
- 120 câu là Gold **DRAFT / tập phát triển đã dùng trước đây**. Nhóm được chốt theo câu hỏi và căn cứ tham chiếu, không chọn vì Graph thắng. AI judge không xác nhận đúng luật. Bằng hoặc kém hơn vẫn giữ.
- Không dùng ZIP `full/without_graph` cũ làm đối chứng thuần: phép cũ tắt thêm các kênh Graph khác. Chúng chỉ phục vụ phân tích trace lịch sử.

Xem README cùng thư mục để ghép hai ZIP và đọc kết quả từng nhóm/từng câu.

**Gold ZIP tự phục hồi:** notebook có sẵn đúng ZIP Gold120 v0.2 gốc. Nếu Kaggle giải nén ZIP lồng trong code Dataset, notebook khôi phục bản gốc và kiểm SHA; không cần thêm Dataset Gold hay file .b64.txt.


In [ ]:
from pathlib import Path
import re
REPO_URL='https://github.com/tvqui/phap-dien-ai-legalgraphrag.git'
CODE_REF='origin/main'  # use the printed commit SHA for a later session
ROOT=Path('/kaggle/working/phap-dien-ai-legalgraphrag')
WORK=Path('/kaggle/working/graph120_evaluation')
SOURCE='github'  # use the refactored GitHub project; attached_zip is only for a pinned snapshot
CODE_ARCHIVE='auto'  # Graph_Study_Code.zip added as a Kaggle Input
RUN_NAME='graph120_mechanism_v1'
PROFILE='standard'  # smoke = 5 fixed questions; standard = full selected cohort
GRAPH_SCOPE='focused'  # focused = 47 input-selected questions; all120 = entire Gold
ARTIFACT='auto'
LABOR_GOLD='evaluation/gold/PhapDienAI_Gold_120_Labor_v0_2.zip'
BACKEND_GEMINI_MODEL='gemini-3.1-flash-lite'
JUDGE_GEMINI_MODEL='gemini-3.5-flash-lite'
BACKEND_GEMINI_SECRET_NAME='GEMINI_API_KEY'
JUDGE_GEMINI_SECRET_NAME='GEMINI_API_KEY'  # optionally GEMINI_API_KEY.2, .3; no automatic rotation
ARM='graph_on'  # graph_on or graph_off; one arm per Run All
ABLATION_CONFIG=ARM
ACTION='run_and_judge'  # run_and_judge | generate_only | judge_only
MAX_NEW_QUESTIONS=None     # new answer attempts, NOT number of API requests
MAX_NEW_JUDGMENTS=None
CLEANUP_OLLAMA=True    # keep the owned server ready between independent cells
RETRY_ERRORS=False      # intentionally enable after capacity is restored
CHECKPOINT_ARCHIVE=''   # optional downloaded ablation ZIP added as Input
METHODS=('graph_on','graph_off')
if SOURCE not in {'attached_zip','github'}:raise ValueError('Invalid SOURCE')
PHASE='ablation';STAGES=['ablation']
if PROFILE not in {'smoke','standard'}:raise ValueError('Invalid PROFILE')
if not re.fullmatch(r'[A-Za-z0-9_-]+',RUN_NAME):raise ValueError('Invalid RUN_NAME')
for value in [BACKEND_GEMINI_SECRET_NAME,JUDGE_GEMINI_SECRET_NAME]:
    if not isinstance(value,str) or not re.fullmatch(r'GEMINI_API_KEY(?:\.[1-9][0-9]*)?',value):raise ValueError('Use Secret NAME, not key value')
if ABLATION_CONFIG not in METHODS:raise ValueError('Invalid ABLATION_CONFIG')
if ACTION not in {'run_and_judge','generate_only','judge_only'}:raise ValueError('Invalid ACTION')
for limit in [MAX_NEW_QUESTIONS,MAX_NEW_JUDGMENTS]:
    if limit is not None and (type(limit) is not int or limit<1):raise ValueError('Batch limit must be positive or None')
OUT=WORK/RUN_NAME/PROFILE/ABLATION_CONFIG
AUTO_RETRY_ROUNDS = 2  # rounds after the full first pass; 0 disables automatic recovery
RETRY_BACKOFF_SECONDS = 10
if type(AUTO_RETRY_ROUNDS) is not int or not 0<=AUTO_RETRY_ROUNDS<=3:raise ValueError('AUTO_RETRY_ROUNDS must be 0..3')
if type(RETRY_BACKOFF_SECONDS) not in (int,float) or not 0<=RETRY_BACKOFF_SECONDS<=60:raise ValueError('RETRY_BACKOFF_SECONDS must be 0..60')


## 1. Lấy code GitHub và cài môi trường

In [ ]:
import os,sys,subprocess,shutil,json,time,urllib.request
WORK.mkdir(parents=True,exist_ok=True)
def run(args, **kw):
    print('RUN',args[0],flush=True)
    return subprocess.run(args,check=True,**kw)
run(['apt-get','update','-y'])
run(['apt-get','install','-y','git','curl','zstd'])
if SOURCE=='github':
    if not ROOT.exists():run(['git','clone','--branch','main','--single-branch',REPO_URL,str(ROOT)])
    else:
        dirty=subprocess.check_output(['git','status','--porcelain','--untracked-files=no'],cwd=ROOT,text=True)
        if dirty.strip():raise RuntimeError('Tracked code changed; do not overwrite it')
    run(['git','fetch','--prune','origin'],cwd=ROOT)
    run(['git','checkout','--detach',CODE_REF],cwd=ROOT)
    run(['git','log','-1','--oneline'],cwd=ROOT)
else:
    import zipfile,stat,hashlib
    from pathlib import PurePosixPath
    import base64
    # Exact original Gold archive embedded for Kaggle's recursive ZIP extraction.
    EMBEDDED_GOLD_NAME='evaluation/gold/PhapDienAI_Gold_120_Labor_v0_2.zip'
    EMBEDDED_GOLD_SHA256='26c278386a69160b3c1479e9280253c0a24838e4fe7d3f9830b2a5cb2d55a7c3'
    EMBEDDED_GOLD_B64='...'
    input_root=Path('/kaggle/input')
    found=list(input_root.rglob('Graph_Study_Code.zip')) if CODE_ARCHIVE=='auto' else [Path(CODE_ARCHIVE)]
    def digest(data):return hashlib.sha256(data).hexdigest()
    def safe_name(name):
        path=PurePosixPath(name)
        if path.is_absolute() or '..' in path.parts or chr(92) in name or ':' in name:raise RuntimeError('Unsafe code path')
    def copy_verified(name,data,expected):
        safe_name(name)
        if digest(data)!=expected:raise RuntimeError('Code checksum mismatch: '+name)
        p=ROOT/name
        if p.exists() and digest(p.read_bytes())!=expected:raise RuntimeError('Local code changed; use fresh ROOT')
        p.parent.mkdir(parents=True,exist_ok=True);p.write_bytes(data)
    def decode_transport(data,encoded):
        if not encoded:return data
        try:return base64.b64decode(data,validate=True)
        except ValueError:raise RuntimeError('Invalid Gold transport text; upload the supplied .b64.txt unchanged') from None
    ROOT.mkdir(parents=True,exist_ok=True)
    if (ROOT/'.git').exists():raise RuntimeError('Use a fresh session or SOURCE=github; do not overwrite a checkout')
    if len(found)==1 and found[0].is_file():
        with zipfile.ZipFile(found[0]) as z:
            infos=z.infolist()
            if sum(m.file_size for m in infos)>150*1024**2:raise RuntimeError('Code archive too large')
            names=set()
            for m in infos:
                safe_name(m.filename)
                if stat.S_ISLNK(m.external_attr>>16):raise RuntimeError('Unsafe archive symlink')
                if m.filename.casefold() in names:raise RuntimeError('Duplicate archive path')
                names.add(m.filename.casefold())
            inventory=json.loads(z.read('GRAPH_CODE_SHA256.json'))
            actual={m.filename for m in infos};transports={}
            for name in inventory:
                safe_name(name)
                if name in actual:transports[name]=(name,False)
                elif name.endswith('.zip') and name+'.b64.txt' in actual:transports[name]=(name+'.b64.txt',True)
                else:raise RuntimeError('Code archive incomplete: '+name)
            if actual!=({v[0] for v in transports.values()}|{'GRAPH_CODE_SHA256.json'}):raise RuntimeError('Unexpected archive files')
            for name,expected in inventory.items():
                storage,encoded=transports[name]
                copy_verified(name,decode_transport(z.read(storage),encoded),expected)
    elif not found or (len(found)==1 and found[0].is_dir()):
        manifests=list(input_root.rglob('GRAPH_CODE_SHA256.json')) if not found else list(found[0].rglob('GRAPH_CODE_SHA256.json'))
        if len(manifests)!=1:raise RuntimeError('Add exactly one code Dataset; set CODE_ARCHIVE if needed')
        source_root=manifests[0].parent.resolve()
        inventory=json.loads(manifests[0].read_text(encoding='utf-8'))
        transports={};total=0
        for name in inventory:
            safe_name(name)
            direct=(source_root/name).resolve()
            if not direct.is_relative_to(source_root):raise RuntimeError('Unsafe code dataset path')
            encoded=False;path=direct
            if not path.is_file() and name.endswith('.zip'):
                local_transport=Path(str(direct)+'.b64.txt')
                if local_transport.is_file():path=local_transport;encoded=True
                else:
                    supplements=list(input_root.rglob(PurePosixPath(name).name+'.b64.txt'))
                    if len(supplements)==1:path=supplements[0];encoded=True
                    elif name==EMBEDDED_GOLD_NAME and inventory[name]==EMBEDDED_GOLD_SHA256:
                        # Restore only the pinned original ZIP; never rebuild an
                        # archive from extracted files or skip its fingerprint.
                        transports[name]=(None,True)
                        print('Gold ZIP restored from notebook; original SHA-256 remains enforced.',flush=True)
                        continue
                    else:raise RuntimeError('Missing pinned Gold ZIP: '+name+'. The notebook embeds only Gold120 v0.2; use the matching original file for another Gold version.')
            if not path.is_file():raise RuntimeError('Code Dataset is incomplete: '+name)
            if not path.resolve().is_relative_to(input_root.resolve()):raise RuntimeError('Code file outside Kaggle Input')
            total+=path.stat().st_size
            if total>150*1024**2:raise RuntimeError('Code dataset too large')
            transports[name]=(path,encoded)
        for name,expected in inventory.items():
            path,encoded=transports[name]
            data=EMBEDDED_GOLD_B64.encode('ascii') if path is None else path.read_bytes()
            copy_verified(name,decode_transport(data,encoded),expected)
    else:raise RuntimeError('Multiple code ZIPs found; set CODE_ARCHIVE explicitly')
if not (ROOT/'evaluation/graph_qa/study/run.py').exists():raise RuntimeError('New Graph study code missing. Upload code ZIP or push the new folder first.')
if not (ROOT/'evaluation/core/retry_failed.py').exists():
    raise RuntimeError('Push the saved-error retry code to GitHub before running this notebook.')
# Keep Kaggle's CUDA-compatible Torch and isolate project dependencies.
import importlib.metadata
constraints=[]
for package in ('torch','torchvision','torchaudio'):
    try:constraints.append(package+'=='+importlib.metadata.version(package))
    except importlib.metadata.PackageNotFoundError:
        if package=='torch':raise RuntimeError('Kaggle runtime has no Torch; select a GPU image.')
constraint_file=WORK/'torch_constraints.txt'
constraint_file.write_text('\n'.join(constraints)+'\n',encoding='utf-8')
VENV=WORK/'.venv'
run([sys.executable,'-m','venv','--system-site-packages','--without-pip',str(VENV)])
PYTHON=str(VENV/'bin/python')
run([sys.executable,'-m','pip','--python',PYTHON,'install','--disable-pip-version-check',
    '-q','-c',str(constraint_file),'-e',str(ROOT)+'[retrieval,online]','datasets'])
run([PYTHON,'-c','import torch,FlagEmbedding,faiss,datasets; assert torch.cuda.is_available(); print("CUDA dependencies ready:",torch.cuda.device_count())'])
if not (ROOT/'evaluation/suite/protocol.py').exists() or 'labor-evaluation-v2-20261028' not in (ROOT/'evaluation/suite/protocol.py').read_text(encoding='utf-8'):
    raise RuntimeError('Push evaluation suite v2 before running this notebook.')
if not (ROOT/'evaluation/suite/suite.py').exists():
    raise RuntimeError('The evaluation suite must be present at evaluation/suite on GitHub before Run All.')

if not (ROOT/'evaluation/graph_qa/study/run.py').exists():
    raise RuntimeError('Chưa có bộ đo mới trên GitHub. Push folder đo rồi chạy lại cell lấy code.')
run([sys.executable,'-m','pip','--python',PYTHON,'install','--disable-pip-version-check','-q',
     '-c',str(constraint_file),'-r',str(ROOT/'evaluation/core/requirements.txt')])
engine=(ROOT/'evaluation/core/eval_core.py').read_text(encoding='utf8')
if '--max-new-questions' not in engine or '--max-new-judgments' not in engine:
    raise RuntimeError('Push the new ablation evaluation code before running this notebook')
# Stop early if GitHub still supplies the old timeout policy.
if ACTION!='judge_only':
    guard_probe="from pathlib import Path;import inspect;from vn_labor_online.config import load_config;from vn_labor_online.runtime import operation_budget;c=load_config(Path('src/online/backend/config/online_kaggle.yaml'));assert 'minimum_call_seconds' in inspect.signature(operation_budget).parameters;assert c.applicability.context_tokens==c.adjudication.context_tokens==32768;assert c.applicability.minimum_call_seconds==25 and c.applicability.batch_prompt_tokens==7000;assert c.applicability.max_output_tokens==1536;assert c.adjudication.minimum_call_seconds==20;print('TIMEOUT_POLICY_VERIFIED')"
    try:run([PYTHON,'-c',guard_probe],cwd=ROOT)
    except subprocess.CalledProcessError:
        raise RuntimeError('Timeout policy preflight failed; read the Python error above. Generation has not started. Verify code/config before retrying.') from None

if ACTION!='judge_only':
    contract_probe="from pathlib import Path\nimport sys\nsys.path.insert(0,str(Path('evaluation/suite').resolve()))\nfrom qa import DisabledAuditor\nfrom vn_labor_online.pipeline import OnlinePipeline\nfrom vn_labor_online.models import Evidence,EvidenceState\nsource=Evidence(unit_id='synthetic_preflight',text='A synthetic source must retain its identifier.',verified=True,retrieval_method='bm25')\nauditor=DisabledAuditor()\nitems,decisions,_=auditor.audit([source])\npipe=OnlinePipeline.__new__(OnlinePipeline)\npipe.applicability=auditor\npack=pipe._build_evidence_pack('Explain the synthetic source.','2025-01-01',{},EvidenceState(slots={},gaps=[],coverage=1),items,decisions)\nassert len(pack.evidence)==1\nassert pack.evidence[0].applicability.audit_status=='UNRESOLVED'\nassert not pack.evidence[0].applicability.supports_claim\nassert 'EXPERIMENT_APPLICABILITY_NOT_CHECKED' in pack.evidence[0].warnings\npipe.applicability=object()\nassert not pipe._build_evidence_pack('Explain.','2025-01-01',{},EvidenceState(slots={},gaps=[],coverage=1),items,decisions).evidence\nprint('AUDITOR_ABLATION_CONTRACT_VERIFIED')\n"
    run([PYTHON,'-c',contract_probe],cwd=ROOT)


In [ ]:
from kaggle_secrets import UserSecretsClient
try:
    hf_token=UserSecretsClient().get_secret('HF_TOKEN')
    if hf_token:os.environ['HF_TOKEN']=hf_token
except Exception:print('HF_TOKEN optional; public downloads will be attempted.')
print('Gemini credentials are loaded only for the selected generation/judge cell; values are never printed.')

## 2. Khởi động Ollama trên GPU 1

In [ ]:
if ACTION!='judge_only':
    if {'ablation','vlsp'} & set(STAGES):
        # Use GPU 1 for Ollama, GPU 0 for embedding/reranker. Own only the server
        # started by this notebook; never terminate another session's server.
        gpu_lines=subprocess.check_output(['nvidia-smi','--query-gpu=name','--format=csv,noheader'],text=True).splitlines()
        print('GPUs:',gpu_lines,flush=True)
        if len(gpu_lines)<2:raise RuntimeError('Select GPU T4 x2 in Kaggle settings for this configuration.')
        if shutil.which('ollama') is None:
            run(['bash','-lc','set -o pipefail; curl -fsSL https://ollama.com/install.sh | sh'])
        def ollama_json(path):
            with urllib.request.urlopen('http://127.0.0.1:11434'+path,timeout=5) as response:return json.load(response)
        OWN_OLLAMA=globals().get('OWN_OLLAMA')
        previous_log=globals().get('OLLAMA_LOG')
        if previous_log is not None and not previous_log.closed:previous_log.close()
        OLLAMA_LOG=(WORK/'ollama.log').open('a',encoding='utf-8')
        try:
            ollama_json('/api/tags')
            print('Reusing existing Ollama; verify it is not serving another workload.',flush=True)
        except Exception:
            environment=dict(os.environ);environment['CUDA_VISIBLE_DEVICES']='1'
            OWN_OLLAMA=subprocess.Popen(['ollama','serve'],env=environment,stdout=OLLAMA_LOG,stderr=subprocess.STDOUT)
            for attempt in range(60):
                if OWN_OLLAMA.poll() is not None:raise RuntimeError('Ollama stopped; read '+str(WORK/'ollama.log'))
                try:ollama_json('/api/tags');break
                except Exception:time.sleep(2)
            else:raise RuntimeError('Ollama did not become ready')
        models={m.get('name') for m in ollama_json('/api/tags').get('models',[])}
        if 'qwen3:8b' not in models:
            for attempt in range(3):
                try:run(['ollama','pull','qwen3:8b'],timeout=900);break
                except (subprocess.CalledProcessError,subprocess.TimeoutExpired):
                    if attempt==2:raise
                    time.sleep(10)
        def warm_ollama():
            # Fixed synthetic input; no Gold, no legal content. Startup latency
            # is recorded separately and excluded from per-question latency.
            payload={'model':'qwen3:8b','stream':False,'think':False,'keep_alive':'15m',
                'format':{'type':'object','properties':{'ready':{'type':'boolean'}},'required':['ready']},
                'options':{'temperature':0,'num_predict':32,'num_ctx':32768},
                'messages':[{'role':'user','content':'Return a JSON object with ready equal to true.'}]}
            request=urllib.request.Request('http://127.0.0.1:11434/api/chat',
                data=json.dumps(payload).encode('utf-8'),headers={'Content-Type':'application/json'})
            started=time.monotonic()
            try:
                with urllib.request.urlopen(request,timeout=180) as response:
                    result=json.load(response)
            except Exception:
                raise RuntimeError('Ollama preflight failed. Do not retry generation while an old inference may still be running; inspect ollama.log and restart the owned server/session.') from None
            content=json.loads(result.get('message',{}).get('content','{}'))
            if not result.get('done') or result.get('done_reason')=='length' or content!={'ready':True}:
                raise RuntimeError('Ollama preflight output is incomplete or invalid.')
            loaded=[m for m in ollama_json('/api/ps').get('models',[]) if m.get('name',m.get('model'))=='qwen3:8b']
            if not loaded or not loaded[0].get('size_vram',0)>0:
                raise RuntimeError('Qwen is not resident on GPU; inspect GPU/VRAM before starting 120 questions.')
            status={'status':'READY','model':'qwen3:8b','context_tokens':32768,
                'size_vram':loaded[0]['size_vram'],'size':loaded[0].get('size'),
                'elapsed_seconds':time.monotonic()-started,'scope':'SYNTHETIC_STARTUP_NOT_QUERY_LATENCY'}
            (WORK/'ollama_preflight.json').write_text(json.dumps(status,indent=2),encoding='utf-8')
            print('Ollama GPU preflight:',status,flush=True)
            return status
        warm_ollama()
        print('Ollama and model ready.',flush=True)
    
    else:
        OWN_OLLAMA=None;OLLAMA_LOG=None
else:
    OWN_OLLAMA=None;OLLAMA_LOG=None
    print('Judge-only: no Ollama/model-generation startup required.')


## 3. Chuẩn bị hoặc khôi phục checkpoint

In [ ]:
import json,sys,shutil,zipfile,stat
from pathlib import PurePosixPath
if CHECKPOINT_ARCHIVE and not OUT.exists():
    with zipfile.ZipFile(CHECKPOINT_ARCHIVE) as archive:
        if sum(m.file_size for m in archive.infolist())>8*1024**3:raise RuntimeError('Checkpoint too large')
        for m in archive.infolist():
            name=PurePosixPath(m.filename)
            if name.is_absolute() or '..' in name.parts or chr(92) in m.filename or ':' in m.filename or stat.S_ISLNK(m.external_attr>>16):raise RuntimeError('Unsafe checkpoint path')
        OUT.mkdir(parents=True)
        archive.extractall(OUT)
OUT.mkdir(parents=True,exist_ok=True)
BUNDLE=OUT/'bundle120';SETTINGS=OUT/'settings.json'
if ACTION!='judge_only' and (WORK/'ollama_preflight.json').exists():
    shutil.copyfile(WORK/'ollama_preflight.json',OUT/'ollama_preflight.json')
sys.path.insert(0,str(ROOT/'evaluation/suite'))
from common import detect_artifact
artifact=Path(detect_artifact(Path('/kaggle/input'))) if ARTIFACT=='auto' else Path(ARTIFACT)
sys.path.insert(0,str(ROOT/'đo'))
from retry_failed import policy, retry_failed, restart_owned_ollama
config=json.loads((ROOT/'evaluation/core/settings.example.json').read_text(encoding='utf8'))
config['retry_policy']=policy(AUTO_RETRY_ROUNDS,RETRY_BACKOFF_SECONDS)
config.update(online_config=str(ROOT/'src/online/backend/config/online_kaggle.yaml'),cache=str(WORK/'cache'),device='cuda:0')
config['judge']['model']=JUDGE_GEMINI_MODEL
import hashlib
config['graph_study_protocol']='graph-mechanism-120-v1'
config['graph_study_scope']=GRAPH_SCOPE
if GRAPH_SCOPE not in {'focused','all120'}:raise ValueError('Invalid GRAPH_SCOPE')
config['graph_grouping_sha256']=hashlib.sha256((ROOT/'evaluation/graph_qa/study/question_groups.json').read_bytes()).hexdigest()
config['hardware_comparison_scope']='GPU_SPEC'  # same GPU model/VRAM/driver; actual UUIDs remain logged
if SETTINGS.exists() and json.loads(SETTINGS.read_text(encoding='utf8'))!=config:raise RuntimeError('Settings changed; use NEW RUN_NAME')
SETTINGS.write_text(json.dumps(config,ensure_ascii=False,indent=2),encoding='utf8')
selection={'backend_gemini_model':BACKEND_GEMINI_MODEL,'judge_gemini_model':JUDGE_GEMINI_MODEL,'ablation_config':ABLATION_CONFIG}
pinned=OUT/'experiment_config.json'
if pinned.exists() and json.loads(pinned.read_text(encoding='utf8'))!=selection:raise RuntimeError('Model plan changed; use NEW RUN_NAME')
pinned.write_text(json.dumps(selection,indent=2),encoding='utf8')
if not (BUNDLE/'manifest.json').exists():
    run([PYTHON,'-u','evaluation/graph_qa/study/run.py','prepare','--artifact',str(artifact),'--gold',str(ROOT/LABOR_GOLD),'--bundle',str(BUNDLE),'--cache',str(WORK/'cache'),'--allow-draft'],cwd=ROOT)
else:
    frozen=json.loads((BUNDLE/'manifest.json').read_text(encoding='utf8'))
    if Path(frozen['artifact_source']).resolve()!=artifact.resolve():raise RuntimeError('Artifact path changed; do not silently rebind checkpoint')
print('Bundle frozen:',BUNDLE)

run([PYTHON,'-u','evaluation/graph_qa/study/run.py','freeze','--bundle',str(BUNDLE)],cwd=ROOT)


## 4. Hàm điều khiển — không tự gọi model

In [ ]:
from retry_failed import retry_failed, restart_owned_ollama
from contextlib import contextmanager
from IPython.display import FileLink,display

@contextmanager
def model_environment(stage):
    from kaggle_secrets import UserSecretsClient
    label=BACKEND_GEMINI_SECRET_NAME if stage=='generation' else JUDGE_GEMINI_SECRET_NAME
    try:token=UserSecretsClient().get_secret(label)
    except Exception:raise RuntimeError('Enable selected Secret: '+label) from None
    if not isinstance(token,str) or not token.strip():raise RuntimeError('Selected Secret is empty')
    values={'GEMINI_API_KEY':token.strip(),'GEMINI_MODEL':BACKEND_GEMINI_MODEL,
        'VN_LABOR_GEMINI_RPM':'13','VN_LABOR_GEMINI_MAX_RETRIES':'2','VN_LABOR_GEMINI_MAX_WAIT_SECONDS':'70',
        'VN_LABOR_RESEARCHER_API_KEY':None,'VN_LABOR_APPLICABILITY_API_KEY':None,'VN_LABOR_ADJUDICATION_API_KEY':None,
        'VN_LABOR_RESEARCHER_URL':'http://127.0.0.1:11434/api/chat',
        'VN_LABOR_APPLICABILITY_MODE':'hybrid','VN_LABOR_APPLICABILITY_PROVIDER':'ollama',
        'VN_LABOR_APPLICABILITY_MODEL':'qwen3:8b','VN_LABOR_APPLICABILITY_URL':'http://127.0.0.1:11434/api/chat',
        'VN_LABOR_ADJUDICATION_URL':'http://127.0.0.1:11434/api/chat',
        'VN_LABOR_RESEARCHER_MODE':'gemini','VN_LABOR_RESEARCHER_MODEL':BACKEND_GEMINI_MODEL,
        'VN_LABOR_ADJUDICATION_MODE':'ollama','VN_LABOR_ADJUDICATION_MODEL':'qwen3:8b',
        'VN_LABOR_ADJUDICATION_REVIEW_PROVIDER':'gemini','VN_LABOR_ADJUDICATION_REVIEW_MODEL':BACKEND_GEMINI_MODEL}
    previous={name:os.environ.get(name) for name in values}
    try:
        for name,value in values.items():
            if value is None:os.environ.pop(name,None)
            else:os.environ[name]=value
        yield
    finally:
        for name,value in previous.items():
            if value is None:os.environ.pop(name,None)
            else:os.environ[name]=value

def export_checkpoint():
    archive=Path(shutil.make_archive(str(OUT)+'_graph_results','zip',root_dir=OUT))
    display(FileLink(str(archive)))
    return archive

def run_configuration(method):
    if method not in METHODS:raise ValueError('Invalid method')
    if method!=ABLATION_CONFIG or ACTION=='judge_only':
        print('Generation not selected:',method);return
    destination=OUT/'runs'/method
    args=[PYTHON,'-u','evaluation/graph_qa/study/run.py','run','--bundle',str(BUNDLE),'--config',str(SETTINGS),
          '--out',str(destination),'--task','qa','--methods',method,'--repeats','1','--seed','42','--k','10','--allow-draft','--allow-llm']
    if PROFILE=='smoke':args+=['--limit','5']
    if (destination/'run_manifest.json').exists():args+=['--resume']
    if RETRY_ERRORS:args+=['--retry-errors']
    if MAX_NEW_QUESTIONS is not None:args+=['--max-new-questions',str(MAX_NEW_QUESTIONS)]
    try:
        def invoke_generation(command):
            with model_environment('generation'):return subprocess.run(command,cwd=ROOT,check=False)
        result=invoke_generation(args)
        result=retry_failed(destination,args,invoke_generation,result,
            max_rounds=AUTO_RETRY_ROUNDS,backoff_seconds=RETRY_BACKOFF_SECONDS,
            before_retry=lambda failed:restart_owned_ollama(globals(),failed))
        progress=json.loads((destination/'progress.json').read_text(encoding='utf8')) if (destination/'progress.json').exists() else {}
        print(method,'progress:',progress)
        scheduled=(type(progress.get('expected')) is int and progress['expected']>0
            and progress.get('saved')==progress['expected'] and progress.get('paused') is None)
        known_checkpoint=result.returncode==2 and (progress.get('paused')=='BATCH_LIMIT' or scheduled)
        if result.returncode and not known_checkpoint:
            raise RuntimeError('Generation paused/failed; checkpoint preserved. Restore service before retrying ERROR records.')
        if result.returncode==2 and scheduled:
            print('All attempts saved; ERROR records remain. Judge will score only OK records, not treat errors as PASS.',flush=True)
    finally:export_checkpoint()

def judge_configuration(method):
    if method not in METHODS:raise ValueError('Invalid method')
    if method!=ABLATION_CONFIG or ACTION=='generate_only':
        print('Judge not selected:',method);return
    destination=OUT/'runs'/method
    if not (destination/'run_manifest.json').exists():raise RuntimeError('Generate this configuration first')
    args=[PYTHON,'-u','evaluation/graph_qa/study/run.py','judge','--run',str(destination),'--allow-llm']
    if MAX_NEW_JUDGMENTS is not None:args+=['--max-new-judgments',str(MAX_NEW_JUDGMENTS)]
    try:
        def invoke_judge(command):
            with model_environment('judge'):return subprocess.run(command,cwd=ROOT,check=False)
        result=invoke_judge(args)
        result=retry_failed(destination,args,invoke_judge,result,stage='judge',
            max_rounds=AUTO_RETRY_ROUNDS,backoff_seconds=RETRY_BACKOFF_SECONDS)
        if result.returncode:raise RuntimeError('Judge failed; answers and completed scores preserved')
        records=[json.loads(line) for line in (destination/'responses.jsonl').read_text(encoding='utf8').splitlines() if line.strip()]
        print(method,'AI_JUDGED',sum(r.get('judge',{}).get('status')=='AI_JUDGED' for r in records),'of',len(records),'saved attempts')
    finally:export_checkpoint()

## 5. Sinh câu trả lời của nhánh đã chọn


In [ ]:
run_configuration(ABLATION_CONFIG)

## 6. Chấm nhánh đã chọn sau khi sinh xong


In [ ]:
judge_configuration(ABLATION_CONFIG)

## 7. Bảng kết quả riêng và ZIP của cấu hình này

In [ ]:
destination=OUT/'runs'/ABLATION_CONFIG
if (destination/'run_manifest.json').exists():
    run([PYTHON,'-u','evaluation/graph_qa/study/run.py','report','--runs',str(destination),'--out',str(OUT/'comparison')],cwd=ROOT)
    from IPython.display import Markdown
    display(Markdown((OUT/'comparison/comparison.md').read_text(encoding='utf8')))
export_checkpoint()

## 8. Dọn dịch vụ sau cấu hình đã chọn

In [ ]:
if CLEANUP_OLLAMA:
    # Chỉ chạy khi kết thúc toàn bộ phép đo trong phiên này; không chạy giữa các phase.
    if OWN_OLLAMA is not None and OWN_OLLAMA.poll() is None:
        OWN_OLLAMA.terminate()
        try:OWN_OLLAMA.wait(timeout=10)
        except subprocess.TimeoutExpired:
            OWN_OLLAMA.kill(); OWN_OLLAMA.wait()
    if OLLAMA_LOG is not None: OLLAMA_LOG.close()
    print('Đã dọn Ollama do notebook này tạo; không dừng dịch vụ được tái sử dụng.')
else:
    print('Giữ Ollama để chạy cell cấu hình tiếp theo. Đặt CLEANUP_OLLAMA=True rồi chạy cell này khi kết thúc.')


## Ghép hai nhánh sau khi tải về
Dùng run.py compare trên máy, hoặc thêm cả hai ZIP làm Input rồi chạy lệnh tương ứng trong một notebook. Không cần gọi model để ghép kết quả. Xem README.
